## TMC-LM full pipeline on Colab GPU

Runs the same stages as the Docker full pipeline (`README.md` section 7):

1. clone repo to `/content/tmc-llm` and install deps
2. build llama.cpp (CUDA, CPU fallback) for GGUF convert + quantize
3. `dataset_builder` -> `train_lora` (LoRA) -> `merge_lora` -> GGUF F16 + Q4_K_M
4. verify + inference smoke test

**Before you start:**
- Push your local commits to `https://github.com/4r6ie/tmc-llm` first. The notebook clones from GitHub, so uncommitted local changes are NOT included.
- `data/processed/*`, `models/*`, and `external/*` are gitignored, so they are rebuilt/persisted only inside this Colab VM. Use the last cell to download or Drive-backup `models/` before the VM is recycled.
- Training is 5 epochs on a T4: expect ~2-4 h. Use **Runtime > Change runtime type > T4 GPU**.

In [ ]:
!nvidia-smi

In [ ]:
# Clone the repo. If it already exists (kernel restarted mid-run), just reuse it.
import os
if not os.path.exists('/content/tmc-llm/.git'):
    !git clone https://github.com/4r6ie/tmc-llm.git /content/tmc-llm

%cd -q /content/tmc-llm

In [ ]:
# System deps: OCR for scanned docs (dataset step) + build tools for llama.cpp
!apt-get update -qq
!apt-get install -y -qq --no-install-recommends tesseract-ocr tesseract-ocr-eng cmake build-essential > /dev/null

In [ ]:
# Install the package (core deps: torch, transformers, peft, datasets, ...).
# For Colab T4 the default torch wheel already has CUDA, so no index-url override.
!python -m pip install -q -e .

In [ ]:
# Build llama.cpp with CUDA (fallback to CPU if nvcc is missing). This creates
# external/llama.cpp/build/bin/{llama-cli, llama-quantize, ...} used below.
%cd -q /content/tmc-llm

bash_script = r'''
set -e
cd /content/tmc-llm
if [ ! -d external/llama.cpp/build ]; then
    git clone --depth 1 https://github.com/ggml-org/llama.cpp external/llama.cpp
    cd external/llama.cpp
    if command -v nvcc >/dev/null 2>&1; then
        echo "nvcc found - building with CUDA"
        cmake -B build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON
    else
        echo "nvcc not found - building CPU-only llama.cpp"
        cmake -B build -DCMAKE_BUILD_TYPE=Release
    fi
    cmake --build build --config Release -j$(nproc)
else
    echo "llama.cpp already built - skipping"
fi
test -x external/llama.cpp/build/bin/llama-cli || { echo 'llama-cli missing after build'; exit 1; }
'''
import subprocess
subprocess.run(['bash', '-c', bash_script], check=True)

In [ ]:
# 1) Build the dataset from raw sources -> data/processed/{train,validation,test}.jsonl
%cd -q /content/tmc-llm
!python -m tmc_llm.dataset_builder --source-dir data/raw/tmc_sources --output-dir data/processed

In [ ]:
# 2) LoRA fine-tune TinyLlama-1.1B-Chat-v1.0 (fp16 on T4, 5 epochs)
#    -> models/adapters/tmc-lm-tinyllama-lora-v1.0
%cd -q /content/tmc-llm
!python -m tmc_llm.train_lora --config configs/train_lora.yaml

In [ ]:
# 3) Merge LoRA adapter into the base model
#    -> models/merged/tmc-lm-tinyllama-v1.0
%cd -q /content/tmc-llm
!python -m tmc_llm.merge_lora --base-model TinyLlama/TinyLlama-1.1B-Chat-v1.0 --adapter-dir models/adapters/tmc-lm-tinyllama-lora-v1.0 --output-dir models/merged/tmc-lm-tinyllama-v1.0

In [ ]:
# 4) Convert merged HF model -> F16 GGUF (llama.cpp, needs the gguf python pkg)
%cd -q /content/tmc-llm
!python -m pip install -q gguf
!mkdir -p models/gguf
!python external/llama.cpp/convert_hf_to_gguf.py models/merged/tmc-lm-tinyllama-v1.0 --outfile models/gguf/tmc-lm-tinyllama-f16.gguf --outtype f16

In [ ]:
# 5) Quantize F16 -> Q4_K_M for lightweight CPU inference
%cd -q /content/tmc-llm
!external/llama.cpp/build/bin/llama-quantize models/gguf/tmc-lm-tinyllama-f16.gguf models/gguf/tmc-lm-tinyllama-q4_k_m.gguf Q4_K_M

In [ ]:
# 6) Verify the quantized GGUF
%cd -q /content/tmc-llm
!python -m tmc_llm.gguf_check --path models/gguf/tmc-lm-tinyllama-q4_k_m.gguf

In [ ]:
# 7) Inference smoke test via the llama.cpp binary that failed before.
%cd -q /content/tmc-llm
!external/llama.cpp/build/bin/llama-cli -m models/gguf/tmc-lm-tinyllama-q4_k_m.gguf -c 2048 --temp 0.2 --repeat-penalty 1.12 -n 128 -p "What is TMC's vision?"

### Save your work before the VM is recycled

`models/`, `data/processed/`, and `external/` are gitignored and only live in this VM. Export them before disconnecting.

In [ ]:
# Option A: download the GGUF + adapter as a zip
%cd -q /content
!zip -qr tmc-llm-models.zip tmc-llm/models/gguf tmc-llm/models/merged tmc-llm/models/adapters
from google.colab import files
files.download('/content/tmc-llm-models.zip')

In [ ]:
# Option B (alternative to the cell above): mirror everything to Google Drive.
# Uncomment the lines below, authorize, then re-run the training cells if needed.
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/tmc-llm
# !cp -r /content/tmc-llm/models /content/tmc-llm/data/processed /content/drive/MyDrive/tmc-llm/